Converting parsed data to natural language

Reading from our theorem_parsed.json

In [2]:
import json

Checking we can print out our data

In [70]:
# Open the JSON file
with open('./external/txt/parsed data/theorem_parsed.json') as f:
    data = json.load(f)

# Print the data (it will be stored as a Python dictionary)
print(data)

{'problem #0': {'axiom': 'axio(b)&(~b|~c)', 'conjecture': 'conj(~a|b)&(b|c)', 'ac axiom': {'ac axiom #0': {'language': 'cnf', 'name': 'i_0_2', 'type': 'plain', 'logical formula': '(b)'}, 'ac axiom #1': {'language': 'cnf', 'name': 'i_0_6', 'type': 'negated_conjecture', 'logical formula': '(a)'}}, 'cnfrefutation': {'refutation step #0': {'language': 'fof', 'name': "'conj(~a|b)&(b|c)'", 'type': 'conjecture', 'logical formula': '((~(a)|b)&(b|c))'}, 'refutation step #1': {'language': 'fof', 'name': "'axio(b)&(~b|~c)'", 'type': 'axiom', 'logical formula': '(b&(~(b)|~(c)))'}, 'refutation step #2': {'language': 'fof', 'name': 'c_0_2', 'type': 'negated_conjecture', 'logical formula': '~(((~a|b)&(b|c)))', 'source': {'inference #0': {'name': 'fof_simplification', 'type': 'status(thm)', 'name of premises': "inference(assume_negation,[status(cth)],['conj(~a|b)&(b|c)'])"}, 'inference #1': {'name': 'assume_negation', 'type': 'status(cth)', 'name of premises': "'conj(~a|b)&(b|c)'"}}}, 'refutation step

Processing parsed theorem proof steps into natural language.

Example of what we want:
* Given the axiom a, can you prove the conjecture (a|~b)? Answer: Step1) Begin by assuming the negation of the conjecture which is simplified to ~((a|~b)). Step 2) Converting the result of Step 1 to negated normal form simplifies to (~a&b). Step 3) Applying split conjunction to Step 2 simplifies to (~-a). Step 4) Our axiom a can be rewritten as a. Stpe 5) Given step 4 produces a and step 3 produces ~a we have derived a contradiction. Therefore our initial conjecture is proven. So yes, we can prove it.

In [265]:
amt_of_problems = len(data)

#for every problem
for x in range(amt_of_problems):
    ref_step_info = ""
    prompt = ""
    
    axiom = "Given axiom " + str(data[f'problem #{x}']['axiom'] + ", ")
    conjecture = "can you prove the conjecture " + str(data[f'problem #{x}']["conjecture"] + "? ")
    
    total_ref_steps = len(data[f'problem #{x}']['cnfrefutation'])
    starting_ref_step = 0

    #Sometimes, the first or first two steps do not need to be accounted for.
    if data[f'problem #{x}']['cnfrefutation']["refutation step #0"]["type"] == "axiom" or data[f'problem #{x}']['cnfrefutation']["refutation step #0"]["type"] == "conjecture":
        starting_ref_step = starting_ref_step + 1
    if data[f'problem #{x}']['cnfrefutation']["refutation step #1"]["type"] == "axiom" or data[f'problem #{x}']['cnfrefutation']["refutation step #1"]["type"] == "conjecture":
        starting_ref_step = starting_ref_step + 1 

    #for every refutation step in the current problem
    for y in range(starting_ref_step, total_ref_steps):
        current_ref_step = data[f'problem #{x}']['cnfrefutation'][f'refutation step #{y}'] #type str
        current_language = current_ref_step["language"] #type str
        current_type = current_ref_step["type"] #type str
        current_logical_formula = current_ref_step["logical formula"] #type str
        current_src = current_ref_step["source"] #type object
        total_infer = len(current_src)
        
        current_step = 0
        if (y == 2 and starting_ref_step == 2):
            current_step = y-1
            ref_step_info = ref_step_info + f"step {current_step}) Begin by "
        elif (y == 1 and starting_ref_step ==1):
            current_step = y
            ref_step_info = ref_step_info + f"step {current_step}) Begin by "
        else:
            current_step = y-1
            ref_step_info = ref_step_info + f"step {current_step}) "

        #If there is more than zero inferences in src of the current ref step
        if total_infer > 0:
            #If there are nested inferences
            if total_infer > 1:

                #for every inference in the current refutation step of the current problem
                for z in range(total_infer, 0, -1):
                    current_infer = current_src[f"inference #{z-1}"] #type object
                    current_infer_name = current_infer["name"] #type str
                    current_infer_type = current_infer["type"] #type str
                    current_infer_name_of_prem = current_infer["name of premises"] #type str
                    #if we are looking at the innermost theorem
                    if (current_step == 1):
                        if (z == total_infer):
                            #print("True", current_infer_name, total_infer)
                            if current_infer_name == "assume_negation":
                                ref_step_info = ref_step_info + "assuming the negation of the conjecture "
                            if current_infer_name == "fof_simplification":
                                ref_step_info = ref_step_info + "assuming the simplification of the conjecture"
                        elif (z != total_infer):
                            if current_infer_name == "fof_simplification":
                                ref_step_info = ref_step_info + f"which is simplified to " + current_logical_formula + ". "
                            if current_infer_name == "assume_negation":
                                ref_step_info = ref_step_info + f"which is negated to" + current_logical_formula + ". "
                                
                    elif (current_step != 1):
                        #ref_step_info = ref_step_info + current_infer_name
                        
                        if (z == total_infer):
                            if current_infer_name == "fof_simplification":
                                ref_step_info = ref_step_info + f"Converting the result of step {current_step-1} which is simplified to " + current_logical_formula + ". "
                            if current_infer_name == "assume_negation":
                                ref_step_info = ref_step_info + f"Converting the result of step {current_step-1} which is negated to" + current_logical_formula + ". "
                            

                        if (z == total_infer) and (total_infer == 3):
                            if current_infer_name == "fof_nnf":
                                ref_step_info = ref_step_info + f"Converting the result of step {current_step-1} to negated normal form "
                                
                            

                        if (z != total_infer) and (total_infer == 3):
                            if current_infer_name == "fof_nnf":
                                ref_step_info = ref_step_info + f"and converted to negated normal form "
                        
                        if (z == 1):
                            if current_infer_name == "distribute":
                                ref_step_info = ref_step_info + f"which is distributed to " + current_logical_formula + ". "
                        if current_infer_name == "rw":
                            ref_step_info = ref_step_info + f"Given step {current_step-1} produced something differently for our axiom and the first few initial steps did not, we have derived at a contradiction"
                        
                        
                                
            #If there is only one inference
            if total_infer == 1:
                current_infer = current_src["inference #0"] #type object
                current_infer_name = current_infer["name"] #type str
                current_infer_type = current_infer["type"] #type str
                current_infer_name_of_prem = current_infer["name of premises"] #type str
                if current_step == 1:
                    if current_infer_name == "fof_simplification":
                        ref_step_info = ref_step_info + "assuming the simplification of the conjecture "+ current_logical_formula + ". "
                else:
                    if current_infer_name == "fof_simplification":
                        ref_step_info = ref_step_info + f"Applying simplification to the result of step {current_step-1} provides us with " + current_logical_formula + ". "
                    if current_infer_name == "split_conjunct":
                        ref_step_info = ref_step_info + f"Using split conjunction on to the result of step {current_step-1} provides us with " + current_logical_formula + ". "
                    if current_infer_name == "fof_nnf":
                        ref_step_info = ref_step_info + f"Applying negated normal form to the result of step {current_step-1} provides us with " + current_logical_formula + ". "
                    if current_infer_name == "cn":
                        ref_step_info = ref_step_info + f"Now we take the negated conjecture of step {current_step-1} matches what we had as our conjecture where we now obtain" + current_logical_formula + ". "
                    #ref_step_info = ref_step_info + "test1 "
            
                
    print("\n" + axiom + conjecture + ref_step_info) #This is a test. 
    print("=====")


Given axiom axio(b)&(~b|~c), can you prove the conjecture conj(~a|b)&(b|c)? step 1) Begin by assuming the negation of the conjecture which is simplified to ~(((~a|b)&(b|c))). step 2) Converting the result of step 1 to negated normal form and converted to negated normal form which is distributed to (((~b|a)&(~c|a))&((~b|~b)&(~c|~b))). step 3) Applying simplification to the result of step 2 provides us with (b&(~b|~c)). step 4) Using split conjunction on to the result of step 3 provides us with (~b|~b). step 5) Applying negated normal form to the result of step 4 provides us with (b&(~b|~c)). step 6) Now we take the negated conjecture of step 5 matches what we had as our conjecture where we now obtain(~b). step 7) Using split conjunction on to the result of step 6 provides us with (b). step 8) Given step 7 produced something differently for our axiom and the first few initial steps did not, we have derived at a contradiction
=====

Given axiom axio(~a|~d)&(a), can you prove the conjectu

In [179]:
for z in range(3, 0, -1):
    print(z-1)

2
1
0
